In [2]:
import re
import os
import glob
import pandas as pd

In [6]:
TRANSCRIPT_PATH = "dca_d1_1.txt"

In [7]:
with open(TRANSCRIPT_PATH, "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

print(raw_text[:3000])

((TAPE-HEADER "DCA TAPE 2; WASHINGTON NATIONAL ATCT; DEPARTURE RADAR CONTROLLER (DR1); 118.95 MHZ; MAY 26, 1992, 1524 EDT; TRANSCRIBER JLO"))

((COMMENT "POSITION WAS APPARENTLY COMBINED WITH DR1 HIGH; SEVERAL AIRCRAFT TRANSMITTING ON ANOTHER FREQUENCY; POSITION WAS LATER DECOMBINED; CONTROLLER CHANGES; POSITION WORKED DEPARTURES OFF ANDREWS AFB, AND DAVISON ARMY AIRFIELD AS WELL AS NATIONAL; DUE TO PROXIMITY OF DULLES AND BALTIMORE, MOST DEPARTURES WERE TRANSFERRED TO ONE OF THESE FACILITIES RATHER THAN BEING CHANGED TO THE CENTER AS IN PREVIOUS FACILITIES"))


((FROM DR1-1)
(TO DAL209)
(TEXT   DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGHT ZERO)
(TIMES    63.04    66.01))
  

((FROM DAL209)
(TO DR1-1)
(TEXT   LEFT TO TWO EIGHTY DELTA TWO OH NINE)
(TIMES    66.65    69.08))

((FROM AAL1581)
(TO DR1-1)
(TEXT   APPROACH AMERICAN EIGHT AH FIFTEEN EIGHTY ONE IS WITH YOU OUT OF TWO)
(TIMES    93.55    96.11))

((FROM DR1-1)
(TO AAL1581)
(TEXT   AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPAR

In [8]:
message_pattern = re.compile(
    r"\(\(FROM\s+(.*?)\)\s*"
    r"\(TO\s+(.*?)\)\s*"
    r"\(TEXT\s+(.*?)\)\s*"
    r"\(TIMES\s+([0-9.]+)\s+([0-9.]+)\)",
    re.DOTALL
)

matches = message_pattern.findall(raw_text)
len(matches)

397

In [9]:
rows = []
for frm, to, text, t1, t2 in matches:
    rows.append({
        "from": frm.strip(),
        "to": to.strip(),
        "text": text.strip(),
        "start_time": float(t1),
        "end_time": float(t2),
        "duration_s": float(t2) - float(t1)
    })

df_msgs = pd.DataFrame(rows)
df_msgs.head(20)

,from,to,text,start_time,end_time,duration_s
0,DR1-1,DAL209,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...,63.04,66.01,2.97
1,DAL209,DR1-1,LEFT TO TWO EIGHTY DELTA TWO OH NINE,66.65,69.08,2.43
2,AAL1581,DR1-1,APPROACH AMERICAN EIGHT AH FIFTEEN EIGHTY ONE ...,93.55,96.11,2.56
3,DR1-1,AAL1581,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...,96.63,101.06,4.43
4,AAL1581,DR1-1,UP TO ONE SEVEN THOUSAND AMERICAN FIFTEEN EIGH...,101.30,103.46,2.16
5,DR1-1,DAL209,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO,106.51,109.48,2.97
6,DAL209,DR1-1,LEFT TO TWO FORTY DELTA TWO OH NINE,109.85,112.06,2.21
7,DR1-1,DAL209,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...,195.95,201.46,5.51
8,DAL209,DR1-1,OKAY WE'LL GO DIRECT LINDEN RIGHT NOW DELTA TW...,201.98,204.77,2.79
9,DR1-1,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,205.16,208.22,3.06


In [10]:
def clean_atc_text(text):
    text = text.upper()
    text = re.sub(r"\(UNINTELLIGIBLE.*?\)", " UNINTELLIGIBLE ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_msgs["clean_text"] = df_msgs["text"].apply(clean_atc_text)
df_msgs[["from", "to", "clean_text"]].head(20)

,from,to,clean_text
0,DR1-1,DAL209,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...
1,DAL209,DR1-1,LEFT TO TWO EIGHTY DELTA TWO OH NINE
2,AAL1581,DR1-1,APPROACH AMERICAN EIGHT AH FIFTEEN EIGHTY ONE ...
3,DR1-1,AAL1581,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...
4,AAL1581,DR1-1,UP TO ONE SEVEN THOUSAND AMERICAN FIFTEEN EIGH...
5,DR1-1,DAL209,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO
6,DAL209,DR1-1,LEFT TO TWO FORTY DELTA TWO OH NINE
7,DR1-1,DAL209,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...
8,DAL209,DR1-1,OKAY WE'LL GO DIRECT LINDEN RIGHT NOW DELTA TW...
9,DR1-1,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...


In [11]:
df_ctrl = df_msgs[df_msgs["from"].str.startswith("DR", na=False)].copy()
df_ctrl[["from", "to", "clean_text"]].head(20)

,from,to,clean_text
0,DR1-1,DAL209,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...
3,DR1-1,AAL1581,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...
5,DR1-1,DAL209,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO
7,DR1-1,DAL209,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...
9,DR1-1,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...
11,DR1-1,DAL209,DELTA TWO ZERO NINE CONTACT DULLES APPROACH CO...
13,DR1-1,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...
16,DR1-1,N99G,AERO STAR NINE NINE GOLF FLY HEADING THREE THR...
18,DR1-1,DAL743,DELTA SEVEN FORTY THREE WASHINGTON DEPARTURE C...
19,DR1-1,N99G,AERO STAR NINE NINE GOLF CLIMB AND MAINTAIN AH...


In [12]:
ENTITY_TYPES = [
    "CALLSIGN",
    "ACTION",
    "RUNWAY",
    "RUNWAY_HEADING",
    "HEADING",
    "ALTITUDE",
    "FLIGHT_LEVEL",
    "WAYPOINT",
    "FREQUENCY",
    "SPEED",
    "DIRECTION",
    "NAV_INSTRUCTION"
]

ENTITY_TYPES

['CALLSIGN',
 'ACTION',
 'RUNWAY',
 'RUNWAY_HEADING',
 'HEADING',
 'ALTITUDE',
 'FLIGHT_LEVEL',
 'WAYPOINT',
 'FREQUENCY',
 'SPEED',
 'DIRECTION',
 'NAV_INSTRUCTION']

In [13]:
patterns = {
    "HEADING": r"\bHEADING\s+((?:ZERO|ONE|TWO|THREE|FOUR|FIVE|SIX|SEVEN|EIGHT|NINER|NINE|OH|\s)+)\b",
    "ALTITUDE": r"\b(?:MAINTAIN|CLIMB AND MAINTAIN|DESCEND AND MAINTAIN)\s+((?:FLIGHT LEVEL\s+)?(?:ZERO|ONE|TWO|THREE|FOUR|FIVE|SIX|SEVEN|EIGHT|NINER|NINE|OH|\s)+)\b",
    "RUNWAY": r"\bRUNWAY\s+([0-9]{1,2}[LRC]?)\b",
    "WAYPOINT": r"\bDIRECT\s+([A-Z0-9]{3,10})\b",
    "FREQUENCY": r"\bONE\s+TWO[0-9A-Z\s]*POINT\s+[0-9A-Z\s]+\b|\b[0-9]{3}\.[0-9]{1,2}\b",
    "DIRECTION": r"\b(LEFT|RIGHT)\b"
}

action_keywords = [
    "TURN LEFT",
    "TURN RIGHT",
    "FLY HEADING",
    "CLIMB AND MAINTAIN",
    "DESCEND AND MAINTAIN",
    "MAINTAIN",
    "PROCEED DIRECT",
    "CONTACT",
    "RESUME YOUR OWN NAVIGATION",
    "JOIN",
    "EXPECT",
    "RADAR CONTACT",
    "DO NOT EXCEED",
    "EXPEDITE"
]

In [14]:
def weak_extract_entities(text):
    entities = {
        "ACTION": [],
        "HEADING": [],
        "ALTITUDE": [],
        "RUNWAY": [],
        "WAYPOINT": [],
        "FREQUENCY": [],
        "DIRECTION": []
    }

    for action in action_keywords:
        if action in text:
            entities["ACTION"].append(action)

    for label, pattern in patterns.items():
        matches = re.findall(pattern, text)
        if matches:
            entities[label] = matches if isinstance(matches, list) else [matches]

    return entities

df_ctrl["weak_entities"] = df_ctrl["clean_text"].apply(weak_extract_entities)
df_ctrl[["clean_text", "weak_entities"]].head(20)

,clean_text,weak_entities
0,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...,"{'ACTION': ['TURN LEFT'], 'HEADING': ['TWO EIG..."
3,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...,"{'ACTION': ['CLIMB AND MAINTAIN', 'MAINTAIN', ..."
5,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO,"{'ACTION': ['TURN LEFT'], 'HEADING': ['TWO FOU..."
7,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...,"{'ACTION': ['TURN RIGHT', 'RESUME YOUR OWN NAV..."
9,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,"{'ACTION': ['TURN LEFT'], 'HEADING': ['TWO NIN..."
11,DELTA TWO ZERO NINE CONTACT DULLES APPROACH CO...,"{'ACTION': ['CONTACT'], 'HEADING': [], 'ALTITU..."
13,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,"{'ACTION': ['TURN LEFT'], 'HEADING': ['TWO THR..."
16,AERO STAR NINE NINE GOLF FLY HEADING THREE THR...,"{'ACTION': ['FLY HEADING', 'CLIMB AND MAINTAIN..."
18,DELTA SEVEN FORTY THREE WASHINGTON DEPARTURE C...,"{'ACTION': ['CLIMB AND MAINTAIN', 'MAINTAIN'],..."
19,AERO STAR NINE NINE GOLF CLIMB AND MAINTAIN AH...,"{'ACTION': ['CLIMB AND MAINTAIN', 'MAINTAIN'],..."


In [15]:
df_ctrl["actions"] = df_ctrl["weak_entities"].apply(lambda x: x["ACTION"])
df_ctrl["headings"] = df_ctrl["weak_entities"].apply(lambda x: x["HEADING"])
df_ctrl["altitudes"] = df_ctrl["weak_entities"].apply(lambda x: x["ALTITUDE"])
df_ctrl["runways"] = df_ctrl["weak_entities"].apply(lambda x: x["RUNWAY"])
df_ctrl["waypoints"] = df_ctrl["weak_entities"].apply(lambda x: x["WAYPOINT"])
df_ctrl["frequencies"] = df_ctrl["weak_entities"].apply(lambda x: x["FREQUENCY"])
df_ctrl["directions"] = df_ctrl["weak_entities"].apply(lambda x: x["DIRECTION"])

df_ctrl[[
    "clean_text",
    "actions",
    "headings",
    "altitudes",
    "runways",
    "waypoints",
    "frequencies",
    "directions"
]].head(30)

,clean_text,actions,headings,altitudes,runways,waypoints,frequencies,directions
0,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...,[TURN LEFT],[TWO EIGHT ZERO],[],[],[],[],[LEFT]
3,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...,"[CLIMB AND MAINTAIN, MAINTAIN, CONTACT, RADAR ...",[],[ONE SEVEN ],[],[],[],[]
5,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO,[TURN LEFT],[TWO FOUR ZERO],[],[],[],[],[LEFT]
7,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...,"[TURN RIGHT, RESUME YOUR OWN NAVIGATION]",[TWO SEVEN ZERO ],[],[],[LINDEN],[],[RIGHT]
9,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,[TURN LEFT],[TWO NINER ZERO ],[],[],[],[],[LEFT]
11,DELTA TWO ZERO NINE CONTACT DULLES APPROACH CO...,[CONTACT],[],[],[],[],[],[]
13,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,[TURN LEFT],[TWO THREE ZERO ],[],[],[],[],[LEFT]
16,AERO STAR NINE NINE GOLF FLY HEADING THREE THR...,"[FLY HEADING, CLIMB AND MAINTAIN, MAINTAIN]",[THREE THREE ZERO ],[FOUR ],[],[],[],[]
18,DELTA SEVEN FORTY THREE WASHINGTON DEPARTURE C...,"[CLIMB AND MAINTAIN, MAINTAIN]",[],[FLIGHT LEVEL TWO THREE ZERO],[],[],[],[]
19,AERO STAR NINE NINE GOLF CLIMB AND MAINTAIN AH...,"[CLIMB AND MAINTAIN, MAINTAIN]",[],[],[],[],[],[]


In [16]:
df_ctrl["callsign"] = df_ctrl["to"].str.strip().str.upper()
df_ctrl[["callsign", "clean_text"]].head(20)

,callsign,clean_text
0,DAL209,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...
3,AAL1581,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...
5,DAL209,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO
7,DAL209,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...
9,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...
11,DAL209,DELTA TWO ZERO NINE CONTACT DULLES APPROACH CO...
13,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...
16,N99G,AERO STAR NINE NINE GOLF FLY HEADING THREE THR...
18,DAL743,DELTA SEVEN FORTY THREE WASHINGTON DEPARTURE C...
19,N99G,AERO STAR NINE NINE GOLF CLIMB AND MAINTAIN AH...


In [17]:
training_df = df_ctrl[[
    "callsign",
    "clean_text",
    "actions",
    "headings",
    "altitudes",
    "runways",
    "waypoints",
    "frequencies",
    "directions"
]].copy()

training_df.head(30)

,callsign,clean_text,actions,headings,altitudes,runways,waypoints,frequencies,directions
0,DAL209,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...,[TURN LEFT],[TWO EIGHT ZERO],[],[],[],[],[LEFT]
3,AAL1581,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...,"[CLIMB AND MAINTAIN, MAINTAIN, CONTACT, RADAR ...",[],[ONE SEVEN ],[],[],[],[]
5,DAL209,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO,[TURN LEFT],[TWO FOUR ZERO],[],[],[],[],[LEFT]
7,DAL209,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...,"[TURN RIGHT, RESUME YOUR OWN NAVIGATION]",[TWO SEVEN ZERO ],[],[],[LINDEN],[],[RIGHT]
9,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,[TURN LEFT],[TWO NINER ZERO ],[],[],[],[],[LEFT]
11,DAL209,DELTA TWO ZERO NINE CONTACT DULLES APPROACH CO...,[CONTACT],[],[],[],[],[],[]
13,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,[TURN LEFT],[TWO THREE ZERO ],[],[],[],[],[LEFT]
16,N99G,AERO STAR NINE NINE GOLF FLY HEADING THREE THR...,"[FLY HEADING, CLIMB AND MAINTAIN, MAINTAIN]",[THREE THREE ZERO ],[FOUR ],[],[],[],[]
18,DAL743,DELTA SEVEN FORTY THREE WASHINGTON DEPARTURE C...,"[CLIMB AND MAINTAIN, MAINTAIN]",[],[FLIGHT LEVEL TWO THREE ZERO],[],[],[],[]
19,N99G,AERO STAR NINE NINE GOLF CLIMB AND MAINTAIN AH...,"[CLIMB AND MAINTAIN, MAINTAIN]",[],[],[],[],[],[]
